In [127]:
import pandas as pd
import numpy as np

In [128]:
df = pd.read_csv('Absenteeism_preprocessed.csv')
df.head()

,Reason_1,Reason_2,Reason_3,Reason_4,Month,Day of Week,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets,Absenteeism Time in Hours
0,False,False,False,True,7,1,289,36,33,239.554,30,0,2,1,4
1,False,False,False,False,7,1,118,13,50,239.554,31,0,1,0,0
2,False,False,False,True,7,2,179,51,38,239.554,31,0,0,0,2
3,True,False,False,False,7,3,279,5,39,239.554,24,0,2,0,4
4,False,False,False,True,7,3,289,36,33,239.554,30,0,2,1,2


In [129]:
df["Absenteeism Time in Hours"].median()

3.0

In [130]:
Targets = np.where(df["Absenteeism Time in Hours"] > 
                   df["Absenteeism Time in Hours"].median(), 1, 0)

In [131]:
df["Targets"] = Targets
df.drop("Absenteeism Time in Hours", axis=1, inplace=True)
df.head()

,Reason_1,Reason_2,Reason_3,Reason_4,Month,Day of Week,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets,Targets
0,False,False,False,True,7,1,289,36,33,239.554,30,0,2,1,1
1,False,False,False,False,7,1,118,13,50,239.554,31,0,1,0,0
2,False,False,False,True,7,2,179,51,38,239.554,31,0,0,0,0
3,True,False,False,False,7,3,279,5,39,239.554,24,0,2,0,1
4,False,False,False,True,7,3,289,36,33,239.554,30,0,2,1,0


In [132]:
df["Targets"].sum()/len(df["Targets"])

np.float64(0.45571428571428574)

In [133]:
df_prepared = df.copy()

In [134]:
df_prepared.head()

,Reason_1,Reason_2,Reason_3,Reason_4,Month,Day of Week,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets,Targets
0,False,False,False,True,7,1,289,36,33,239.554,30,0,2,1,1
1,False,False,False,False,7,1,118,13,50,239.554,31,0,1,0,0
2,False,False,False,True,7,2,179,51,38,239.554,31,0,0,0,0
3,True,False,False,False,7,3,279,5,39,239.554,24,0,2,0,1
4,False,False,False,True,7,3,289,36,33,239.554,30,0,2,1,0


In [135]:
unscaled_inputs = df_prepared.iloc[:, :-1]

In [136]:
#Standardization

In [137]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(unscaled_inputs)

,copy,True
,with_mean,True
,with_std,True


In [138]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns, copy=True, with_mean=True, with_std=True):
        self.scaler = StandardScaler(copy=copy, with_mean= with_mean,with_std= with_std)
        self.columns = columns
        self.mean_ = None
        self.var_ = None

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns], y)
        self.mean_ = np.mean(X[self.columns])
        self.var_ = np.var(X[self.columns])
        return self

    def transform(self, X, y=None):
        init_col_order = X.columns
        X_scaled = pd.DataFrame(self.scaler.transform(X[self.columns]), columns=self.columns)
        X_not_scaled = X.loc[:, ~X.columns.isin(self.columns)]
        return pd.concat([X_scaled, X_not_scaled], axis=1)[init_col_order] 

In [139]:
unscaled_inputs.columns.values

array(['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Month',
       'Day of Week', 'Transportation Expense', 'Distance to Work', 'Age',
       'Daily Work Load Average', 'Body Mass Index', 'Education',
       'Children', 'Pets'], dtype=object)

In [140]:
columns_to_scale = ['Month',
       'Day of Week', 'Transportation Expense', 'Distance to Work', 'Age',
       'Daily Work Load Average', 'Body Mass Index',
       'Children', 'Pets']

In [141]:
absenteeism_scaler = CustomScaler(columns_to_scale)
absenteeism_scaler.fit(unscaled_inputs)
scaled_inputs = absenteeism_scaler.transform(unscaled_inputs)
scaled_inputs.head()

c:\Users\wizzi\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:4266: FutureWarning: The behavior of DataFrame.var with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return var(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


,Reason_1,Reason_2,Reason_3,Reason_4,Month,Day of Week,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets
0,False,False,False,True,0.182726,-0.683704,1.005844,0.412816,-0.536062,-0.806331,0.767431,0,0.880469,0.268487
1,False,False,False,False,0.182726,-0.683704,-1.574681,-1.141882,2.130803,-0.806331,1.002633,0,-0.019280,-0.589690
2,False,False,False,True,0.182726,-0.007725,-0.654143,1.426749,0.248310,-0.806331,1.002633,0,-0.919030,-0.589690
3,True,False,False,False,0.182726,0.668253,0.854936,-1.682647,0.405184,-0.806331,-0.643782,0,0.880469,-0.589690
4,False,False,False,True,0.182726,0.668253,1.005844,0.412816,-0.536062,-0.806331,0.767431,0,0.880469,0.268487


In [142]:
#scaled_inputs = scaler.transform(unscaled_inputs)

In [143]:
from sklearn.model_selection import train_test_split

In [144]:
x_train, x_test, y_train, y_test = train_test_split(scaled_inputs, Targets, test_size=0.2, random_state=20)

In [145]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

In [146]:
reg = LogisticRegression()
reg.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [147]:
reg.score(x_train, y_train)

0.775

In [148]:
model_output = reg.predict(x_train) 

In [149]:
y_train

array([0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0,

In [150]:
np.sum(model_output == y_train)/len(model_output)

np.float64(0.775)

In [151]:
reg.coef_

array([[ 2.80136327e+00,  9.33540824e-01,  3.09673857e+00,
         8.57183147e-01,  1.66403124e-01, -8.43159241e-02,
         6.13215559e-01, -7.77871894e-03, -1.65545282e-01,
        -7.68487792e-05,  2.71154773e-01, -2.06026920e-01,
         3.61897667e-01, -2.85728905e-01]])

In [152]:
reg.intercept_

array([-1.65662792])

In [153]:
unscaled_inputs.columns.values

array(['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Month',
       'Day of Week', 'Transportation Expense', 'Distance to Work', 'Age',
       'Daily Work Load Average', 'Body Mass Index', 'Education',
       'Children', 'Pets'], dtype=object)

In [154]:
features = unscaled_inputs.columns.values

In [155]:
summary_table = pd.DataFrame(columns=["Features"], data=features)
summary_table["Weights"] = np.transpose(reg.coef_)
summary_table.sort_values(by="Weights", ascending=False)

,Features,Weights
2,Reason_3,3.096739
0,Reason_1,2.801363
1,Reason_2,0.933541
3,Reason_4,0.857183
6,Transportation Expense,0.613216
12,Children,0.361898
10,Body Mass Index,0.271155
4,Month,0.166403
9,Daily Work Load Average,-0.000077
7,Distance to Work,-0.007779


In [156]:
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table.sort_index(inplace=True)


In [158]:
summary_table["Odds_ratio"] = np.exp(summary_table.Weights)
summary_table.sort_values(by="Odds_ratio", ascending=False)

,Features,Weights,Odds_ratio
3,Reason_3,3.096739,22.125672
1,Reason_1,2.801363,16.467081
2,Reason_2,0.933541,2.543499
4,Reason_4,0.857183,2.356513
7,Transportation Expense,0.613216,1.846359
13,Children,0.361898,1.436052
11,Body Mass Index,0.271155,1.311478
5,Month,0.166403,1.181049
10,Daily Work Load Average,-0.000077,0.999923
8,Distance to Work,-0.007779,0.992251


In [159]:
#test

In [160]:
reg.score(x_test, y_test)

0.7428571428571429

In [161]:
predicted_proba = reg.predict_proba(x_test)
predicted_proba[:, 1]

array([0.26155478, 0.39169248, 0.59067835, 0.19499225, 0.92680175,
       0.68010859, 0.68684516, 0.8666185 , 0.202816  , 0.24714545,
       0.51778013, 0.80371403, 0.92152605, 0.2936134 , 0.69342299,
       0.42955722, 0.45850356, 0.42797923, 0.61841309, 0.95143761,
       0.3022326 , 0.20409466, 0.60490399, 0.57736104, 0.73368852,
       0.24390833, 0.48938568, 0.13200488, 0.79780626, 0.21350927,
       0.37375472, 0.68661431, 0.68825923, 0.54144395, 0.20409466,
       0.5080899 , 0.21062014, 0.74428002, 0.43679853, 0.59059004,
       0.22487149, 0.43478641, 0.21699119, 0.39320712, 0.81427639,
       0.5706047 , 0.69235426, 0.27269735, 0.20223705, 0.18048174,
       0.59250079, 0.34584097, 0.66753175, 0.28570041, 0.84967583,
       0.47073808, 0.8892122 , 0.25604441, 0.31941989, 0.31737343,
       0.72172164, 0.65693939, 0.31194258, 0.78717355, 0.19838182,
       0.26524577, 0.08189923, 0.2301838 , 0.72734148, 0.33451142,
       0.21060789, 0.29495458, 0.90900579, 0.43914262, 0.61975

In [162]:
#saving the model

In [163]:
import pickle

In [164]:
with open("model", "wb") as file:
    pickle.dump(reg, file)

In [165]:
with open("scaler", "wb") as file:
    pickle.dump(absenteeism_scaler, file)